In [1]:
# ============================================================
# PART 1: INSTALL & IMPORTS
# ============================================================

!pip install -q gradio --upgrade
!pip install -q langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -q "transformers>=4.38.0" accelerate
!pip install -q rank_bm25
!pip install -q rouge-score
!pip install -q numpy pandas

import os
import re
import gc
import time
import numpy as np
import torch
import gradio as gr
from pathlib import Path
from collections import deque

print(f"✅ Gradio: {gr.__version__}")

from kaggle_secrets import UserSecretsClient
secrets  = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"]                 = HF_TOKEN
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN
print(f"✅ HF Token: {HF_TOKEN[:8]}...")
print("✅ Part 1 Done!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 MB 47.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.

In [38]:
# ============================================================
# RAG PIPELINE — As-Sunnah Foundation Dataset
# Part 2: Improved Semantic Chunking
# Part 3: Hybrid Retrieval (FAISS + BM25 + CrossEncoder)
#
# Install before running:
#   pip install langchain langchain-community langchain-text-splitters
#              langchain-huggingface sentence-transformers
#              rank_bm25 faiss-cpu numpy
# ============================================================

import re
import copy
import unicodedata
import numpy as np

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

DATASET_PATH = "/kaggle/input/datasets/mdfaishalahmedrudroo/maindataset/as_sunnah_foundation_extract.txt"


# ╔══════════════════════════════════════════════════════════════╗
# ║              PART 2 — SEMANTIC CHUNKING                     ║
# ╚══════════════════════════════════════════════════════════════╝

# ── 1. Load & clean ───────────────────────────────────────────

def load_and_clean_text(filepath: str) -> str:
    with open(filepath, "r", encoding="utf-8") as f:
        raw = f.read()
    text = raw.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = "\n".join(line.rstrip() for line in text.split("\n"))
    text = unicodedata.normalize("NFC", text)
    return text.strip()


# ── 2. Numeric / stat chunk extractor ────────────────────────
#
#  For sections known to contain key statistics, extract the
#  sentences that carry numbers as a dedicated atomic chunk.
#  This prevents the LLM from having to hunt through a long
#  paragraph for figures, and keeps faithfulness = 1.0.

NUMERIC_PATTERNS = [
    (
        "Iftār Distribution",
        "Iftar Distribution — Year-by-Year Statistics",
        r"(12[,\s]*030|10[,\s]*300|9[,\s]*870|49[,\s]*739)",
        "numeric_stat"
    ),
    (
        "Emergency Relief",
        "Emergency Relief — Impact Statistics",
        r"(৳300|299\s*402|3\s*000 houses|91 livestock|100 auto)",
        "numeric_stat"
    ),
    (
        "Self‑Reliance Project",
        "Self-Reliance Project — Impact Statistics",
        r"5\s*140\s*families",
        "numeric_stat"
    ),
    (
        "Dawāh Activities",
        "Dawah Activities — Impact Statistics",
        r"(256|30 million|30\s*000 questions|10\s*000.*consult)",
        "numeric_stat"
    ),
]


def extract_numeric_chunks(section_content: str,
                            section_name: str,
                            page: str) -> list:
    chunks = []
    for (match_section, label, pattern, category) in NUMERIC_PATTERNS:
        if match_section.lower() not in section_name.lower():
            continue
        sentences   = re.split(r'(?<=[.?!])\s+', section_content)
        stat_sents  = [s for s in sentences
                       if re.search(pattern, s, re.IGNORECASE)]
        if stat_sents:
            stat_text = " ".join(stat_sents)
            chunks.append({
                "page":     page,
                "section":  label,
                "content":  stat_text,
                "category": category
            })
    return chunks


# ── 3. Donation-fund splitter ─────────────────────────────────
#
#  The Home Page lists several distinct donation funds.
#  Splitting each into its own chunk means the retriever can
#  pinpoint individual fund purposes instead of returning the
#  whole paragraph.

DONATION_FUNDS = [
    "Regular Donation Fund",
    "Masjid Complex",
    "Islamic Center",
    "Skill Development Institute",
    "Sadaqah Jariyah Fund",
    "Zakat Fund",
]


def split_donation_funds(body: str, page: str) -> list:
    pattern = re.compile(
        r'(' + '|'.join(re.escape(f) for f in DONATION_FUNDS) + r')',
        re.IGNORECASE
    )
    parts = pattern.split(body)
    if len(parts) <= 1:
        return [{
            "page":     page,
            "section":  "Donation Funds",
            "content":  body.strip(),
            "category": "donation"
        }]

    chunks = []
    i = 1
    while i < len(parts):
        fund_name = parts[i].strip()
        desc      = parts[i + 1].strip() if i + 1 < len(parts) else ""
        if fund_name or desc:
            combined = f"{fund_name}: {desc}".strip(": ")
            chunks.append({
                "page":     page,
                "section":  f"Donation Fund — {fund_name}",
                "content":  combined,
                "category": "donation"
            })
        i += 2

    return chunks or [{
        "page":     page,
        "section":  "Donation Funds",
        "content":  body.strip(),
        "category": "donation"
    }]


# ── 4. Category tagger ────────────────────────────────────────

def infer_category(page: str, section: str) -> str:
    text = (page + " " + section).lower()
    if any(k in text for k in ["skill", "course", "madrasa", "scholar",
                                "training", "education", "student"]):
        return "education"
    if any(k in text for k in ["dawah", "da'wah", "video", "book",
                                "lecture", "shariah", "quran"]):
        return "dawah"
    if any(k in text for k in ["relief", "flood", "iftar", "qurbani",
                                "winter", "tube well", "self-reliance",
                                "self‑reliance", "sick", "orphan",
                                "disaster", "rehabilitation"]):
        return "service"
    if any(k in text for k in ["donation", "fund", "zakat", "sadaqah"]):
        return "donation"
    if any(k in text for k in ["about", "principle", "value",
                                "goal", "objective", "income",
                                "expenditure", "policy"]):
        return "about"
    return "general"


# ── 5. Section parser ─────────────────────────────────────────

def parse_sections(text: str) -> list:
    """
    Level 1 : **Page Heading**
    Level 2 : numbered activity items inside Activities
    Level 3 : *Goal heading* inside About Us

    Returns list of section dicts:
        { page, section, content, category }

    Special handling:
    - Summary page → skipped  (too generic; causes faithfulness = 0)
    - Home Page    → Donation Funds block split per fund
    - About Us     → Goals block split per Education / Da'wah / Service
    - Activities   → each numbered item is its own section
                     + atomic numeric chunk if applicable
    """
    sections = []

    top_pattern = re.compile(r'\n\*\*([^*\n]+)\*\*\n', re.MULTILINE)
    top_splits  = top_pattern.split(text)

    # Text before first heading
    if top_splits[0].strip():
        sections.append({
            "page":     "Introduction",
            "section":  "Introduction",
            "content":  top_splits[0].strip(),
            "category": "general"
        })

    i = 1
    while i + 1 < len(top_splits):
        page_heading = top_splits[i].strip()
        body         = top_splits[i + 1].strip()

        # ── Skip Summary ──────────────────────────────────────
        if page_heading.strip().lower() == "summary":
            i += 2
            continue

        # ── Home Page: split Donation Funds block ─────────────
        if page_heading == "Home Page" and "Donation" in body:
            donation_match = re.search(
                r'(A\s+"Donation Funds".*?)(?=\n\nThe home page also|$)',
                body, re.DOTALL | re.IGNORECASE
            )
            if donation_match:
                before = body[:donation_match.start()].strip()
                donation_block = donation_match.group(1).strip()
                after  = body[donation_match.end():].strip()

                if before:
                    sections.append({
                        "page":     page_heading,
                        "section":  "Home Page — Overview",
                        "content":  before,
                        "category": "general"
                    })
                for fc in split_donation_funds(donation_block, page_heading):
                    sections.append(fc)
                if after:
                    sections.append({
                        "page":     page_heading,
                        "section":  "Home Page — Engagement & Blogs",
                        "content":  after,
                        "category": "general"
                    })
                i += 2
                continue

        # ── Activities: split by numbered items ───────────────
        if "Activities" in page_heading or "Activity" in page_heading:
            num_pattern = re.compile(
                r'\n(\d+)\.\s+\*\*(.+?)\*\*\s*[–—-]\s*', re.MULTILINE
            )
            num_splits = num_pattern.split(body)

            if num_splits[0].strip():
                sections.append({
                    "page":     page_heading,
                    "section":  page_heading,
                    "content":  num_splits[0].strip(),
                    "category": "general"
                })

            j = 1
            while j + 2 < len(num_splits):
                act_title = num_splits[j + 1].strip()
                act_body  = num_splits[j + 2].strip()
                if act_body:
                    cat = infer_category(page_heading, act_title)
                    sections.append({
                        "page":     page_heading,
                        "section":  act_title,
                        "content":  act_body,
                        "category": cat
                    })
                    # Atomic numeric chunk (if applicable)
                    for nc in extract_numeric_chunks(
                            act_body, act_title, page_heading):
                        sections.append(nc)
                j += 3
            i += 2
            continue

        # ── About Us: split goals by heading ─────────────────
        if "About Us" in page_heading:
            goal_pattern = re.compile(
                r'\*(Education goals|Daʼwah goals|Service goals)\*',
                re.IGNORECASE
            )
            goal_splits = goal_pattern.split(body)

            if goal_splits[0].strip():
                sections.append({
                    "page":     page_heading,
                    "section":  "About Us — Overview & Principles",
                    "content":  goal_splits[0].strip(),
                    "category": "about"
                })

            k = 1
            while k + 1 < len(goal_splits):
                goal_title = goal_splits[k].strip()
                goal_body  = goal_splits[k + 1].strip()
                if goal_body:
                    sections.append({
                        "page":     page_heading,
                        "section":  f"About Us — {goal_title}",
                        "content":  goal_body,
                        "category": infer_category(page_heading, goal_title)
                    })
                k += 2
            i += 2
            continue

        # ── Default ───────────────────────────────────────────
        cat = infer_category(page_heading, page_heading)
        sections.append({
            "page":     page_heading,
            "section":  page_heading,
            "content":  body,
            "category": cat
        })
        i += 2

    return sections


# ── 6. Document creator ───────────────────────────────────────

def create_semantic_documents(text: str) -> list:
    """
    <=700 chars  → single Document (no split)
    >700  chars  → RecursiveCharacterTextSplitter with overlap

    Metadata per chunk:
        chunk_id, page, section, category,
        sub_chunk, char_count, original_chunk
    """
    sections = parse_sections(text)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size      = 600,
        chunk_overlap   = 120,
        length_function = len,
        separators      = ["\n\n", "\n", ". ", "! ", "? ", ", ", " ", ""]
    )

    documents = []
    chunk_id  = 0

    for sec in sections:
        content  = sec["content"]
        page     = sec["page"]
        section  = sec["section"]
        category = sec.get("category", "general")

        if len(content) <= 700:
            documents.append(Document(
                page_content = content,
                metadata     = {
                    "chunk_id":       chunk_id,
                    "page":           page,
                    "section":        section,
                    "category":       category,
                    "sub_chunk":      "1/1",
                    "char_count":     len(content),
                    "original_chunk": content,
                    "source":         "as_sunnah_foundation_extract.txt"
                }
            ))
            chunk_id += 1
        else:
            sub_chunks = splitter.split_text(content)
            total      = len(sub_chunks)
            for idx, sub in enumerate(sub_chunks):
                documents.append(Document(
                    page_content = sub,
                    metadata     = {
                        "chunk_id":       chunk_id,
                        "page":           page,
                        "section":        section,
                        "category":       category,
                        "sub_chunk":      f"{idx+1}/{total}",
                        "char_count":     len(sub),
                        "original_chunk": sub,
                        "source":         "as_sunnah_foundation_extract.txt"
                    }
                ))
                chunk_id += 1

    return documents


# ── Run Part 2 ────────────────────────────────────────────────

print("=" * 65)
print("PART 2 — SEMANTIC CHUNKING")
print("=" * 65)

print("📄 Loading dataset...")
raw_text  = load_and_clean_text(DATASET_PATH)
print(f"✅ Raw text: {len(raw_text):,} characters")

documents = create_semantic_documents(raw_text)
print(f"✅ Total chunks created: {len(documents)}")

print("\n📊 Chunk breakdown:")
for doc in documents:
    sub = doc.metadata.get("sub_chunk", "1/1")
    cat = doc.metadata.get("category", "")
    print(f"   [{doc.metadata['chunk_id']:02d}] "
          f"[{cat:<12}] "
          f"{doc.metadata['page']:<30} | "
          f"{doc.metadata['section']:<45} | "
          f"sub:{sub} | {doc.metadata['char_count']} chars")

print("\n🔍 Verification:")
numeric_chunks  = [d for d in documents if d.metadata["category"] == "numeric_stat"]
donation_chunks = [d for d in documents if d.metadata["category"] == "donation"]
summary_chunks  = [d for d in documents if "Summary" in d.metadata["page"]]
print(f"   Numeric/stat chunks        : {len(numeric_chunks)}")
print(f"   Donation fund chunks       : {len(donation_chunks)}")
print(f"   Summary chunks (must be 0) : {len(summary_chunks)}")

print("\n📋 Sample — Iftar numeric chunk:")
for d in numeric_chunks:
    if "Iftar" in d.metadata["section"] or "Iftār" in d.metadata["section"]:
        print(d.page_content)
        break

print("\n✅ Part 2 Complete!\n")


# ╔══════════════════════════════════════════════════════════════╗
# ║    PART 3 — HYBRID RETRIEVAL (FAISS + BM25 + CrossEncoder)  ║
# ╚══════════════════════════════════════════════════════════════╝
#
# Key fixes vs original broken code:
#   ✅ EnsembleRetriever import removed — manually merged
#   ✅ multilingual-e5-base "passage: " / "query: " prefixes correct
#   ✅ BM25 built on original text (no e5 prefix)
#   ✅ CrossEncoder reranks on original_chunk, not prefixed text
#   ✅ original_chunk stored in metadata from Part 2 — LLM only
#      sees dataset content → hallucination near 0
#   ✅ build_rag_prompt() enforces "only from context" instruction

try:
    from langchain_huggingface import HuggingFaceEmbeddings
    print("✅ Using langchain_huggingface")
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings
    print("✅ Using langchain_community.embeddings (fallback)")

from langchain_community.vectorstores import FAISS
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

print("=" * 65)
print("PART 3 — HYBRID RETRIEVAL  (FAISS + BM25 + CrossEncoder)")
print("=" * 65)

# ── Embedding model ───────────────────────────────────────────
print("\n🔄 Loading embedding model (multilingual-e5-base)...")
embedding_model = HuggingFaceEmbeddings(
    model_name    = "intfloat/multilingual-e5-base",
    model_kwargs  = {"device": "cpu"},
    encode_kwargs = {"normalize_embeddings": True}
)
print("✅ Embedding model loaded")

# ── FAISS index ───────────────────────────────────────────────
# multilingual-e5-base requires "passage: " prefix during indexing.
# We deepcopy each doc so the original text is NOT overwritten —
# original_chunk in metadata stays clean for the LLM.

print("\n🔄 Building FAISS index...")
e5_documents = []
for doc in documents:
    d = copy.deepcopy(doc)
    d.page_content = "passage: " + doc.page_content
    e5_documents.append(d)

vectorstore = FAISS.from_documents(
    documents = e5_documents,
    embedding = embedding_model
)
print(f"✅ FAISS index built: {len(e5_documents)} vectors")

# ── BM25 index ────────────────────────────────────────────────
# Uses ORIGINAL text (no "passage: " prefix) for exact keyword match.
print("\n🔄 Building BM25 index...")
bm25_corpus = [doc.page_content.lower().split() for doc in documents]
bm25_index  = BM25Okapi(bm25_corpus)
print("✅ BM25 index built")

# ── CrossEncoder reranker ─────────────────────────────────────
print("\n🔄 Loading CrossEncoder reranker...")
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    max_length = 512
)
print("✅ CrossEncoder loaded")


# ── Core retrieval function ───────────────────────────────────

def hybrid_retrieve(query: str, top_k: int = 6) -> list:
    """
    Step 1 : FAISS semantic search     → top-10 candidates
    Step 2 : BM25 keyword search       → top-10 candidates
    Step 3 : Weighted score merge      → alpha=0.6 FAISS, 0.4 BM25
    Step 4 : CrossEncoder rerank       → final top_k

    Returns list of Documents with extra metadata:
        retrieval_score  (CrossEncoder score — higher = more relevant)
        hybrid_score     (weighted FAISS + BM25)
    """
    # FAISS: "query: " prefix required for multilingual-e5-base
    query_e5      = "query: " + query
    faiss_results = vectorstore.similarity_search_with_score(query_e5, k=10)

    # chunk_id → (doc, l2_distance)
    faiss_map = {
        doc.metadata["chunk_id"]: (doc, score)
        for doc, score in faiss_results
    }

    # BM25: plain lowercase tokens
    bm25_scores = bm25_index.get_scores(query.lower().split())
    top_bm25    = np.argsort(bm25_scores)[::-1][:10]

    # Merge by chunk_id
    candidates = {}

    # Add FAISS candidates  (convert L2 distance → similarity score)
    for chunk_id, (doc, faiss_dist) in faiss_map.items():
        sim = 1.0 / (1.0 + faiss_dist)
        candidates[chunk_id] = {"doc": doc, "faiss": sim, "bm25": 0.0}

    # Add / update BM25 candidates (normalise to [0, 1])
    max_bm25 = float(bm25_scores[top_bm25[0]]) \
               if bm25_scores[top_bm25[0]] > 0 else 1.0
    for idx in top_bm25:
        chunk_id  = documents[idx].metadata["chunk_id"]
        bm25_norm = float(bm25_scores[idx]) / max_bm25
        if chunk_id in candidates:
            candidates[chunk_id]["bm25"] = bm25_norm
        else:
            # Present in BM25 but not in FAISS top-10
            candidates[chunk_id] = {
                "doc":   e5_documents[idx],
                "faiss": 0.0,
                "bm25":  bm25_norm
            }

    # Weighted hybrid score
    alpha = 0.6   # FAISS (semantic) weight
    for c in candidates.values():
        c["hybrid_score"] = alpha * c["faiss"] + (1 - alpha) * c["bm25"]

    # Keep top-12 for CrossEncoder reranking
    sorted_cands = sorted(
        candidates.values(),
        key     = lambda x: x["hybrid_score"],
        reverse = True
    )[:12]

    # CrossEncoder reranks on original_chunk text (clean, no e5 prefix)
    pairs = [
        (query,
         c["doc"].metadata.get("original_chunk", c["doc"].page_content))
        for c in sorted_cands
    ]
    ce_scores = reranker.predict(pairs)

    for idx, c in enumerate(sorted_cands):
        c["ce_score"] = float(ce_scores[idx])

    reranked = sorted(
        sorted_cands,
        key     = lambda x: x["ce_score"],
        reverse = True
    )[:top_k]

    # Build final document list with scores attached
    result_docs = []
    for c in reranked:
        doc = copy.deepcopy(c["doc"])
        doc.metadata["retrieval_score"] = round(c["ce_score"], 4)
        doc.metadata["hybrid_score"]    = round(c["hybrid_score"], 4)
        result_docs.append(doc)

    return result_docs


# ── Context builder ───────────────────────────────────────────

def build_rag_context(query: str, top_k: int = 5) -> str:
    """
    Retrieve top_k chunks and merge their original_chunk text.
    Each block is labelled with page, section and score so the
    LLM can trace every fact back to the dataset.
    """
    retrieved = hybrid_retrieve(query, top_k=top_k)
    parts = []
    for rank, doc in enumerate(retrieved, 1):
        chunk_text = doc.metadata.get("original_chunk", doc.page_content)
        section    = doc.metadata.get("section",  "")
        page       = doc.metadata.get("page",     "")
        score      = doc.metadata.get("retrieval_score", 0)
        category   = doc.metadata.get("category", "")
        parts.append(
            f"[Chunk {rank} | Page: {page} | Section: {section} "
            f"| Category: {category} | Score: {score:.4f}]\n"
            f"{chunk_text}"
        )
    return "\n\n---\n\n".join(parts)


# ── Hallucination-resistant prompt builder ────────────────────

def build_rag_prompt(query: str, top_k: int = 5) -> str:
    """
    Returns a complete prompt for any LLM.
    The system instruction forces the model to answer ONLY from
    the retrieved context — hallucination near 0.
    """
    context = build_rag_context(query, top_k=top_k)
    prompt = (
        "You are a research assistant for As-Sunnah Foundation.\n"
        "Answer the question using ONLY the context provided below.\n"
        "Do NOT add any information that is not present in the context.\n"
        "If the context does not contain the answer, respond with exactly:\n"
        "  'The dataset does not contain this information.'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        "Answer (detailed, based only on the context above):"
    )
    return prompt


# ── Smoke tests ───────────────────────────────────────────────

TEST_QUERIES = [
    ("Who is the founder of As-Sunnah Foundation?",                 4),
    ("Tell me details about As-Sunnah Skill Development Institute.", 5),
    ("What are the education goals of As-Sunnah Foundation?",       4),
    ("How many iftar packages were distributed year by year?",       3),
    ("What donation funds does As-Sunnah Foundation offer?",        4),
    ("Describe the disaster relief and rehabilitation activities.",  5),
]

print("\n" + "=" * 65)
print("SMOKE TESTS")
print("=" * 65)

for query, k in TEST_QUERIES:
    print(f"\n🔍 Query : {query}")
    retrieved = hybrid_retrieve(query, top_k=k)
    for d in retrieved:
        print(f"   [{d.metadata['chunk_id']:02d}] "
              f"CE={d.metadata['retrieval_score']:+.4f} | "
              f"H={d.metadata['hybrid_score']:.4f} | "
              f"[{d.metadata['category']:<12}] "
              f"{d.metadata['page']} → {d.metadata['section'][:45]}")

print("\n" + "=" * 65)
print("SAMPLE FULL PROMPT  (Skill Development Institute query)")
print("=" * 65)
sample_prompt = build_rag_prompt(
    "Tell me details about As-Sunnah Skill Development Institute.",
    top_k=5
)
print(sample_prompt[:1200], "\n...[truncated]\n")

print("✅ Part 3 Complete!")

print("""
╔══════════════════════════════════════════════════════════════╗
║  USAGE — plug into your LLM                                  ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  prompt = build_rag_prompt("your question here", top_k=5)   ║
║                                                              ║
║  Option A — Gemma / Llama (HuggingFace local)               ║
║    inputs  = tokenizer(prompt, return_tensors="pt",          ║
║                        truncation=True, max_length=2048)     ║
║    outputs = model.generate(**inputs, max_new_tokens=512)    ║
║    answer  = tokenizer.decode(outputs[0],                    ║
║                               skip_special_tokens=True)      ║
║                                                              ║
║  Option B — OpenAI / Anthropic API                           ║
║    response = client.messages.create(                        ║
║        model="...", max_tokens=512,                          ║
║        messages=[{"role":"user","content": prompt}]          ║
║    )                                                         ║
╚══════════════════════════════════════════════════════════════╝
""")

PART 2 — SEMANTIC CHUNKING
📄 Loading dataset...
✅ Raw text: 14,448 characters
✅ Total chunks created: 41

📊 Chunk breakdown:
   [00] [general     ] Introduction                   | Introduction                                  | sub:1/1 | 100 chars
   [01] [general     ] Home Page                      | Home Page                                     | sub:1/7 | 517 chars
   [02] [general     ] Home Page                      | Home Page                                     | sub:2/7 | 86 chars
   [03] [general     ] Home Page                      | Home Page                                     | sub:3/7 | 474 chars
   [04] [general     ] Home Page                      | Home Page                                     | sub:4/7 | 190 chars
   [05] [general     ] Home Page                      | Home Page                                     | sub:5/7 | 482 chars
   [06] [general     ] Home Page                      | Home Page                                     | sub:6/7 | 352 chars
   [07] 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded

🔄 Building FAISS index...
✅ FAISS index built: 41 vectors

🔄 Building BM25 index...
✅ BM25 index built

🔄 Loading CrossEncoder reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ CrossEncoder loaded

SMOKE TESTS

🔍 Query : Who is the founder of As-Sunnah Foundation?
   [08] CE=+8.7693 | H=0.8811 | [about       ] About Us → About Us — Overview & Principles
   [01] CE=+2.5584 | H=0.6425 | [general     ] Home Page → Home Page
   [09] CE=+0.6377 | H=0.5623 | [about       ] About Us → About Us — Overview & Principles
   [00] CE=+0.4722 | H=0.4501 | [general     ] Introduction → Introduction

🔍 Query : Tell me details about As-Sunnah Skill Development Institute.
   [13] CE=+5.1450 | H=0.6651 | [education   ] About Us → About Us — Education goals
   [09] CE=+4.4913 | H=0.8681 | [about       ] About Us → About Us — Overview & Principles
   [07] CE=+3.9188 | H=0.6635 | [general     ] Home Page → Home Page
   [06] CE=+3.2213 | H=0.1812 | [general     ] Home Page → Home Page
   [08] CE=+1.7474 | H=0.4558 | [about       ] About Us → About Us — Overview & Principles

🔍 Query : What are the education goals of As-Sunnah Foundation?
   [08] CE=+6.1643 | H=0.4755 | [about    

In [18]:
# ============================================================
# PART 4: GEMMA 2B-IT LLM
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

def load_gemma():
    model_name = "google/gemma-2-2b-it"
    print("🔄 Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype  = torch.float16 if device == "cuda" else torch.float32
    print(f"📱 Device: {device}")

    print("🔄 Loading Gemma 2B-IT model...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name, token=HF_TOKEN,
        torch_dtype=dtype, device_map="auto",
        low_cpu_mem_usage=True
    )

    pipe = pipeline(
        "text-generation", model=model, tokenizer=tokenizer,
        max_new_tokens=400, do_sample=False,
        temperature=1.0, repetition_penalty=1.15
    )
    print("✅ Gemma 2B-IT loaded!")
    return pipe, tokenizer

llm_pipeline, tokenizer = load_gemma()
print("\n✅ Part 4 Done!")

🔄 Loading tokenizer...
📱 Device: cuda
🔄 Loading Gemma 2B-IT model...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

✅ Gemma 2B-IT loaded!

✅ Part 4 Done!


In [8]:
!pip install openai-whisper


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 14.4 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=9a3a61023c8443d968bcd868fc23e9a1480314459bf8fbfc6f8593cf875f8265
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [20]:
# ============================================================
# PART 5: WHISPER STT — English only
# ============================================================

import whisper
import librosa
import numpy as np 

print("🔄 Loading Whisper base.en model (English only, fast)...")
# base.en is optimised for English — smaller and faster than large-v3
whisper_model = whisper.load_model("base.en")
print("✅ Whisper base.en loaded!")


def transcribe_audio(audio_path: str) -> dict:
    """
    Transcribes an English audio file using Whisper base.en.
    Returns:
        {
          "text":  str,   # transcribed text (empty string on failure)
          "error": str | None
        }
    """
    if audio_path is None:
        return {"text": "", "error": "No audio file provided."}

    try:
        print("  📂 Loading audio...")
        audio, sr = librosa.load(audio_path, sr=16000)
        audio_np  = audio.astype(np.float32)
        duration  = len(audio_np) / 16000
        print(f"  ⏱  Duration: {duration:.1f}s")

        if duration < 0.5:
            return {
                "text":  "",
                "error": "Audio too short — please speak for at least 1 second."
            }
        if duration > 60:
            return {
                "text":  "",
                "error": "Audio too long — please keep it under 60 seconds."
            }

        print("  🎤 Transcribing (English)...")
        result = whisper_model.transcribe(
            audio_np,
            language                   = "en",
            task                       = "transcribe",
            fp16                       = torch.cuda.is_available(),
            temperature                = 0.0,
            beam_size                  = 5,
            condition_on_previous_text = False,
            initial_prompt             = "Question about As-Sunnah Foundation."
        )
        text = result["text"].strip()

        if not text or len(text) < 3:
            return {
                "text":  "",
                "error": "Could not transcribe — please speak clearly and try again."
            }

        print(f"  ✅ Transcribed: {text[:120]}")
        return {"text": text, "error": None}

    except Exception as e:
        print(f"  ❌ Transcription error: {e}")
        return {"text": "", "error": str(e)}


# ── Quick test ───────────────────────────────────────────────
print("✅ Part 5 Done! (audio transcription ready)")

🔄 Loading Whisper base.en model (English only, fast)...
✅ Whisper base.en loaded!
✅ Part 5 Done! (audio transcription ready)


In [21]:
# ============================================================
# PART 6: LSTM CONVERSATION MEMORY
# ============================================================

import torch
import torch.nn as nn

class ConversationLSTM(nn.Module):
    """
    Lightweight LSTM: encodes question history,
    outputs per-turn relevance scores.
    """
    def __init__(self, input_dim: int = 768, hidden_dim: int = 128):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc   = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        scores  = torch.sigmoid(self.fc(out))
        return scores.squeeze()


class ConversationMemory:
    """
    LSTM-based sliding window memory (last MAX_TURNS turns).
    Scores past turns by relevance to current query,
    applies exponential time-decay, returns top-N for prompt.
    """
    MAX_TURNS   = 6
    DECAY_ALPHA = 0.75

    def __init__(self, embed_model):
        self.turns    = deque(maxlen=self.MAX_TURNS)
        self.embed_fn = embed_model
        self.lstm     = ConversationLSTM(input_dim=768).eval()

    def add_turn(self, question: str, answer: str,
                 confidence: float, sources: list):
        self.turns.append({
            "question":   question,
            "answer":     answer,
            "confidence": confidence,
            "sources":    sources
        })

    def get_relevant_context(self, current_query: str,
                              top_n: int = 3) -> list:
        if len(self.turns) == 0:
            return []
        if len(self.turns) < 2:
            return list(self.turns)

        all_questions = [t["question"] for t in self.turns] + [current_query]
        embeddings    = self.embed_fn.embed_documents(all_questions)
        embeddings    = torch.tensor(embeddings, dtype=torch.float32)

        with torch.no_grad():
            scores = self.lstm(embeddings.unsqueeze(0))

        turn_scores   = scores[:-1].numpy()
        n             = len(self.turns)
        decay_weights = np.array(
            [self.DECAY_ALPHA ** (n - 1 - i) for i in range(n)]
        )
        final_scores  = turn_scores * decay_weights
        ranked_idx    = np.argsort(final_scores)[::-1][:top_n]
        ranked_idx    = sorted(ranked_idx)
        return [list(self.turns)[i] for i in ranked_idx]

    def format_for_prompt(self, current_query: str) -> str:
        relevant = self.get_relevant_context(current_query)
        if not relevant:
            return ""
        lines = ["Conversation history (most relevant past turns):"]
        for t in relevant:
            lines.append(f"  Q: {t['question']}")
            lines.append(f"  A: {t['answer'][:200]}...")
        return "\n".join(lines)

    def is_followup(self, query: str) -> bool:
        signals = ["it", "that", "this", "they", "he", "she",
                   "more", "elaborate", "explain", "what about",
                   "and", "also", "tell me more"]
        q = query.lower()
        if len(q.split()) <= 4:
            return True
        return any(q.startswith(sig) for sig in signals)

    def build_query(self, query: str) -> str:
        if not self.is_followup(query) or len(self.turns) == 0:
            return query
        last  = list(self.turns)[-1]
        topic = last["question"]
        return f"{topic} — {query}"

    def clear(self):
        self.turns.clear()

    def __len__(self):
        return len(self.turns)


memory = ConversationMemory(embed_model=embedding_model)
print("✅ LSTM conversation memory ready")
print("✅ Part 6 Done!")

✅ LSTM conversation memory ready
✅ Part 6 Done!


In [31]:
# ============================================================
# PART 7: CONFIDENCE SCORING
# ============================================================


def token_overlap(text1: str, text2: str) -> dict:
    t1 = set(text1.lower().split())
    t2 = set(text2.lower().split())
    if not t1 or not t2:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    common    = t1 & t2
    precision = len(common) / len(t1)
    recall    = len(common) / len(t2)
    f1        = 2 * precision * recall / (precision + recall + 1e-9)
    return {
        "precision": round(precision, 4),
        "recall":    round(recall, 4),
        "f1":        round(f1, 4)
    }


def faithfulness_score(answer: str, context_docs: list) -> float:
    if not answer or not context_docs:
        return 0.0
    context = " ".join(
        doc.metadata.get("original_chunk", doc.page_content)
        for doc in context_docs
    ).lower()
    sentences = [
        s.strip() for s in re.split(r'[.!?]', answer)
        if len(s.strip()) > 10
    ]
    if not sentences:
        return 0.0
    supported = sum(
        1 for s in sentences
        if (len(set(s.lower().split()) & set(context.split()))
            / max(len(s.split()), 1)) >= 0.45
    )
    return round(supported / len(sentences), 4)


def retrieval_confidence(docs: list) -> float:
    if not docs:
        return 0.0
    scores     = [doc.metadata.get("retrieval_score", 0.0) for doc in docs]
    normalised = [1 / (1 + np.exp(-s)) for s in scores]
    return round(float(np.mean(normalised)), 4)


def compute_confidence(answer: str, context_docs: list) -> dict:
    faith    = faithfulness_score(answer, context_docs)
    overlap  = token_overlap(
        answer,
        " ".join(
            d.metadata.get("original_chunk", d.page_content)
            for d in context_docs
        )
    )
    ret_conf = retrieval_confidence(context_docs)
    overall  = round(0.45 * faith + 0.30 * overlap["precision"] + 0.25 * ret_conf, 4)
    return {
        "overall":        overall,
        "faithfulness":   faith,
        "precision":      overlap["precision"],
        "recall":         overlap["recall"],
        "retrieval_conf": ret_conf,
        "hallucination":  round(1.0 - faith, 4),
    }


def confidence_label(score: float) -> tuple:
    if score >= 0.75: return "🟢", "High"
    if score >= 0.50: return "🟡", "Medium"
    if score >= 0.25: return "🟠", "Low"
    return "🔴", "Very low"


print("✅ Part 7 Done!")

✅ Part 7 Done!


In [32]:
# ============================================================
# PART 8: PROMPT BUILDER + RAG CHAIN
# ============================================================

FALLBACK_PHRASES = [
    "i don't know", "i do not know", "not found", "cannot find",
    "no information", "not mentioned", "not in the context",
    "not provided", "based on the provided data"
]


def build_prompt(question: str, context_docs: list,
                 history_text: str = "") -> str:
    context_parts = []
    for i, doc in enumerate(context_docs):
        content = doc.metadata.get("original_chunk", doc.page_content)
        content = content.removeprefix("passage: ").removeprefix("query: ")
        context_parts.append(
            f"[Source {i+1} — {doc.metadata['section']}]\n{content}"
        )
    context_text = "\n\n---\n\n".join(context_parts)

    history_block = (
        f"Previous conversation (use only if relevant):\n{history_text}\n\n"
        if history_text else ""
    )

    return f"""<start_of_turn>user
You are a knowledgeable assistant for As-Sunnah Foundation.
Answer STRICTLY based on the provided context below.

Rules:
1. Answer ONLY using information present in the context.
2. If the answer is not in the context, reply exactly: "I don't know based on the provided data."
3. Be concise, factual, and specific — avoid vague statements.
4. Do NOT fabricate statistics, names, or dates.
5. If a question is a follow-up, use the conversation history to understand what is being asked.

{history_block}Relevant context:
{context_text}

Question: {question}
<end_of_turn>
<start_of_turn>model
"""


def clean_gemma_output(text: str) -> str:
    for tok in ["<end_of_turn>", "<start_of_turn>", "<eos>", "<bos>"]:
        text = text.split(tok)[0]
    return re.sub(r'\s+', ' ', text).strip()


def is_fallback_answer(text: str) -> bool:
    t = text.lower()
    return any(p in t for p in FALLBACK_PHRASES)


def generate_answer(question: str) -> dict:
    """
    Full RAG pipeline:
      1. Expand query with LSTM memory (follow-up detection)
      2. Hybrid retrieve + CrossEncoder rerank
      3. Build prompt with history
      4. Gemma generates answer
      5. Confidence scoring
      6. Update LSTM memory
    """
    # Step 1: query expansion
    expanded_query = memory.build_query(question)
    if expanded_query != question:
        print(f"  🔗 Expanded query: {expanded_query}")

    # Step 2: retrieve
    retrieved_docs = hybrid_retrieve(expanded_query, top_k=5)

    # Step 3: history context
    history_text = memory.format_for_prompt(question)

    # Step 4: generate
    prompt  = build_prompt(question, retrieved_docs, history_text)
    outputs = llm_pipeline(prompt, max_new_tokens=400, return_full_text=False)
    answer  = clean_gemma_output(outputs[0]["generated_text"])

    # Step 5: confidence
    confidence = compute_confidence(answer, retrieved_docs)
    fallback   = is_fallback_answer(answer)

    # Sources
    sources = []
    for doc in retrieved_docs:
        content = doc.metadata.get("original_chunk", doc.page_content)
        content = content.removeprefix("passage: ")
        sources.append({
            "chunk_id":        doc.metadata["chunk_id"],
            "section":         doc.metadata["section"],
            "retrieval_score": doc.metadata.get("retrieval_score", 0.0),
            "preview":         content[:160] + "..."
        })

    # Step 6: update memory
    memory.add_turn(
        question=question, answer=answer,
        confidence=confidence["overall"],
        sources=[s["section"] for s in sources]
    )

    return {
        "answer":        answer,
        "confidence":    confidence,
        "sources":       sources,
        "retrieved":     retrieved_docs,
        "is_fallback":   fallback,
        "query_used":    expanded_query,
        "_original_q":   question
    }


# ── Tests ────────────────────────────────────────────────────
memory.clear()

print("=" * 55)
print("🧪 TEST 1 — Factual question")
print("=" * 55)
r1 = generate_answer("Who is the chairman of As-Sunnah Foundation?")
print(f"Answer: {r1['answer']}")
print(f"Confidence: {r1['confidence']['overall']:.4f}")

print("\n" + "=" * 55)
print("🧪 TEST 2 — Follow-up")
print("=" * 55)
r2 = generate_answer("When was it founded?")
print(f"Expanded query: {r2['query_used']}")
print(f"Answer: {r2['answer']}")

print("\n" + "=" * 55)
print("🧪 TEST 3 — Fallback")
print("=" * 55)
r3 = generate_answer("What is the price of rice in Bangladesh?")
print(f"Answer: {r3['answer']}")
print(f"Is fallback: {r3['is_fallback']}")

memory.clear()   # reset for fresh session
print("\n✅ Part 8 Done!")

Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧪 TEST 1 — Factual question


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: Shaikh Ahmadullah.
Confidence: 0.1817

🧪 TEST 2 — Follow-up
  🔗 Expanded query: Who is the chairman of As-Sunnah Foundation? — When was it founded?
Expanded query: Who is the chairman of As-Sunnah Foundation? — When was it founded?
Answer: 2017

🧪 TEST 3 — Fallback


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: I don't know based on the provided data.
Is fallback: True

✅ Part 8 Done!


In [34]:
# ============================================================
# PART 9: RESPONSE FORMATTER
# ============================================================


def metric_bar(value: float, width: int = 14) -> str:
    value  = max(0.0, min(1.0, float(value)))
    filled = int(value * width)
    return "█" * filled + "░" * (width - filled)


def format_response(result: dict) -> str:
    answer     = result["answer"]
    conf       = result["confidence"]
    sources    = result["sources"]
    fallback   = result["is_fallback"]
    query_used = result["query_used"]
    orig_q     = result.get("_original_q", query_used)

    emoji, label = confidence_label(conf["overall"])

    status_line = (
        "⚠️ **Status:** Answer not found in knowledge base — "
        "try rephrasing your question.\n\n"
        if fallback else ""
    )

    expanded_note = (
        f"\n\n*Query expanded to: \"{query_used}\"*"
        if query_used != orig_q else ""
    )

    conf_block = f"""---
### 📊 Confidence Dashboard

| Metric | Score | Visual |
|--------|------:|--------|
| **Overall** {emoji} | `{conf['overall']:.4f}` ({label}) | `{metric_bar(conf['overall'])}` |
| Faithfulness | `{conf['faithfulness']:.4f}` | `{metric_bar(conf['faithfulness'])}` |
| Precision | `{conf['precision']:.4f}` | `{metric_bar(conf['precision'])}` |
| Retrieval confidence | `{conf['retrieval_conf']:.4f}` | `{metric_bar(conf['retrieval_conf'])}` |
| Hallucination risk | `{conf['hallucination']:.4f}` | `{metric_bar(conf['hallucination'])}` |
"""

    source_rows = "\n".join(
        f"  **[{i+1}]** `{s['section']}` "
        f"(score: `{s['retrieval_score']:.3f}`)\n"
        f"  > {s['preview']}"
        for i, s in enumerate(sources[:4])
    )

    return (
        f"{status_line}"
        f"### 🤖 Answer\n{answer}{expanded_note}\n\n"
        f"---\n### 📚 Sources\n{source_rows}\n\n"
        f"{conf_block}"
    )


def format_history_panel(turns: list) -> str:
    if not turns:
        return "*No conversation history yet.*"
    lines = []
    for i, t in enumerate(list(turns)[-5:], 1):
        emoji, label = confidence_label(t["confidence"])
        lines.append(
            f"**Turn {i}** {emoji} Confidence: `{t['confidence']:.4f}` ({label})\n"
            f"- **Q:** {t['question'][:100]}\n"
            f"- **A:** {t['answer'][:150]}...\n"
            f"- *Sources: {', '.join(t['sources'][:3])}*\n---"
        )
    return "\n".join(lines)


print("✅ Part 9 Done!")

✅ Part 9 Done!


In [35]:
# ============================================================
# PART 10: TEXT TO SPEECH — gTTS (English)
# ============================================================

!pip install -q gTTS

from gtts import gTTS
import tempfile
import os


def text_to_speech(text: str) -> str | None:
    """
    Converts English answer text to an MP3 audio file.
    Returns the file path, or None on failure.
    Skips TTS for fallback / very short answers.
    """
    if not text or len(text.strip()) < 10:
        return None

    # Don't TTS the fallback message — not useful as audio
    if "i don't know based on" in text.lower():
        return None

    try:
        with tempfile.NamedTemporaryFile(
            suffix=".mp3", delete=False, prefix="tts_answer_"
        ) as tmp:
            tmp_path = tmp.name

        tts = gTTS(text=text, lang="en", slow=False)
        tts.save(tmp_path)
        print(f"  🔊 TTS saved: {tmp_path}")
        return tmp_path

    except Exception as e:
        print(f"  ⚠️  TTS error: {e}")
        return None


# ── Quick test ───────────────────────────────────────────────
test_audio = text_to_speech(
    "As-Sunnah Foundation is a non-profit organisation founded in 2017 "
    "by Shaikh Ahmadullah, dedicated to education, dawah, and humanitarian service."
)
print(f"✅ TTS test path: {test_audio}")
print("✅ Part 10 Done!")

  🔊 TTS saved: /tmp/tts_answer_r0w42vz8.mp3
✅ TTS test path: /tmp/tts_answer_r0w42vz8.mp3
✅ Part 10 Done!


In [36]:
# ============================================================
# PART 11: MAIN PIPELINE — returns text response + audio
# ============================================================


def process_query(audio_input, text_input, history_state):
    """
    Accepts audio OR text input (audio takes priority).
    Returns:
        response_md   : str        — formatted Markdown answer
        audio_output  : str | None — path to MP3 file (or None)
        history_state : list       — updated session history
        history_md    : str        — formatted history panel
    """
    history_state = history_state or []

    # ── Step 1: Resolve input ────────────────────────────────
    if audio_input is not None:
        print("🎤 Processing audio input...")
        stt = transcribe_audio(audio_input)

        if stt["error"]:
            err_md = f"❌ **Audio error:** {stt['error']}"
            return err_md, None, history_state, format_history_panel(history_state)

        question   = stt["text"]
        input_type = "voice"
        print(f"  📝 Transcribed: {question}")

    elif text_input and text_input.strip():
        question   = text_input.strip()
        input_type = "text"

    else:
        warn = "⚠️ Please provide audio or type a question."
        return warn, None, history_state, format_history_panel(history_state)

    print(f"\n{'='*50}")
    print(f"📝 Question  : {question}")
    print(f"📥 Input type: {input_type}")
    print(f"🔗 Memory    : {len(memory)} prior turns")

    # ── Step 2: RAG ──────────────────────────────────────────
    print("🔄 Retrieving and generating...")
    result = generate_answer(question)
    result["_original_q"] = question

    print(f"✅ Confidence: {result['confidence']['overall']:.4f}")
    print(f"✅ Fallback  : {result['is_fallback']}")

    # ── Step 3: Format text response ─────────────────────────
    input_icon  = "🎤" if input_type == "voice" else "💬"
    header      = f"{input_icon} **Question:** {question}\n\n"
    response_md = header + format_response(result)

    # ── Step 4: TTS — convert answer to audio ────────────────
    print("🔊 Generating audio response...")
    audio_output = text_to_speech(result["answer"])

    # ── Step 5: Update Gradio history state ──────────────────
    history_state.append({
        "question":   question,
        "answer":     result["answer"],
        "confidence": result["confidence"]["overall"],
        "sources":    [s["section"] for s in result["sources"]]
    })

    history_md = format_history_panel(history_state)
    return response_md, audio_output, history_state, history_md


# ── Smoke test ───────────────────────────────────────────────
memory.clear()
resp_md, audio_path, hist, hist_md = process_query(
    None,
    "What services does As-Sunnah Foundation provide?",
    []
)
print(resp_md[:400])
print(f"\n🔊 Audio path : {audio_path}")
print(f"📋 History    : {len(hist)} entries")
memory.clear()
print("\n✅ Part 11 Done!")

Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📝 Question  : What services does As-Sunnah Foundation provide?
📥 Input type: text
🔗 Memory    : 0 prior turns
🔄 Retrieving and generating...
✅ Confidence: 0.9037
✅ Fallback  : False
🔊 Generating audio response...
  🔊 TTS saved: /tmp/tts_answer_wnh3ffwz.mp3
💬 **Question:** What services does As-Sunnah Foundation provide?

### 🤖 Answer
As-Sunnah Foundation provides education, da'wah, and overall humanitarian welfare services.

---
### 📚 Sources
  **[1]** `About Us — Overview & Principles` (score: `6.581`)
  > The *About Us* page reiterates that As‑Sunnah Foundation is a non‑political, non‑profit service‑oriented organisation registered with the govern

🔊 Audio path : /tmp/tts_answer_wnh3ffwz.mp3
📋 History    : 1 entries

✅ Part 11 Done!


In [37]:
# ============================================================
# PART 12: GRADIO INTERFACE — audio + text in, text + audio out
# ============================================================

import gradio as gr

SAMPLE_QUESTIONS = [
    "What is As-Sunnah Foundation?",
    "Who is the chairman and when was the foundation established?",
    "What is the registration number?",
    "What activities does the foundation carry out?",
    "Tell me about the Iftār Distribution programme.",
    "What is the As-Sunnah Skill Development Institute?",
    "What courses does the Skill Development Institute offer?",
    "How does the Self-Reliance project work?",
    "How many families benefited from the Self-Reliance project?",
    "What is the Emergency Relief and Rehabilitation programme?",
    "How many iftār packages were distributed in 2025?",
    "What are the foundation's goals for education?",
    "What are the foundation's daʼwah goals?",
    "What are the income sources of the foundation?",
    "What does the foundation say about zakat?",
    "What is the Sadaqah Jariyah Fund?",
    "How can I volunteer for As-Sunnah Foundation?",
    "What is the price of rice in Bangladesh?",   # fallback test
]


def gradio_submit(audio_input, text_input, history_state):
    return process_query(audio_input, text_input, history_state)


def clear_all():
    memory.clear()
    return (
        None,                                  # audio_input
        "",                                    # text_input
        "*Your answer will appear here...*",   # response_output
        None,                                  # audio_output
        [],                                    # history_state
        "*No history yet.*"                    # history_panel
    )


with gr.Blocks(
    title="As-Sunnah Foundation — RAG Assistant",
    theme=gr.themes.Soft()
) as demo:

    # ── Header ───────────────────────────────────────────────
    gr.HTML("""
    <div style="
        text-align:center; padding:24px 20px;
        background:linear-gradient(135deg,#1a472a,#2d6a4f);
        border-radius:14px; margin-bottom:24px;">
      <h1 style="color:white;margin:0;font-size:26px;">
        🕌 As-Sunnah Foundation
      </h1>
      <h2 style="color:#a8d5b5;margin:8px 0 0;font-size:17px;">
        RAG-Powered Knowledge Assistant
      </h2>
      <p style="color:#c8e6c9;margin:8px 0 0;font-size:13px;">
        Ask in <strong>English</strong> — voice 🎤 or text 💬<br>
        Get answers as <strong>text + audio</strong> 🔊<br>
        LSTM memory · Hybrid retrieval · Confidence scoring
      </p>
    </div>
    """)

    history_state = gr.State([])

    with gr.Row(equal_height=False):

        # ── Left column: Input ───────────────────────────────
        with gr.Column(scale=1, min_width=300):
            gr.Markdown("### 📥 Input")

            audio_input = gr.Audio(
                label="🎤 Voice input — mic or upload (English only)",
                sources=["microphone", "upload"],
                type="filepath"
            )

            gr.HTML("""
            <div style="
                text-align:center;
                color:var(--color-text-secondary);
                font-size:13px; margin:6px 0;">
              — or type below —
            </div>
            """)

            text_input = gr.Textbox(
                label="💬 Text input",
                placeholder="e.g. Who founded As-Sunnah Foundation?",
                lines=3, max_lines=6
            )

            with gr.Row():
                submit_btn = gr.Button(
                    "🔍 Get Answer", variant="primary", size="lg"
                )
                clear_btn = gr.Button(
                    "🗑️ Clear All", variant="secondary"
                )

            gr.Markdown("---")
            gr.Markdown("#### 💡 Sample Questions")
            for q in SAMPLE_QUESTIONS:
                gr.Button(q, size="sm").click(
                    fn=lambda x=q: x,
                    outputs=text_input
                )

        # ── Right column: Output ─────────────────────────────
        with gr.Column(scale=2):
            gr.Markdown("### 📤 Output")

            # ── Text answer ──────────────────────────────────
            response_output = gr.Markdown(
                value="*Your answer will appear here...*",
                label="Answer, Sources & Confidence"
            )

            # ── Audio answer ─────────────────────────────────
            audio_output = gr.Audio(
                label="🔊 Audio answer",
                type="filepath",
                autoplay=True,       # auto-plays when answer arrives
                interactive=False
            )

            # ── History ──────────────────────────────────────
            with gr.Accordion(
                "📜 Conversation History (LSTM memory — last 5 turns)",
                open=False
            ):
                history_panel = gr.Markdown(value="*No history yet.*")
                gr.Button("🔄 Refresh").click(
                    fn=lambda h: format_history_panel(h),
                    inputs=[history_state],
                    outputs=[history_panel]
                )

    # ── Footer ────────────────────────────────────────────────
    gr.HTML("""
    <div style="
        text-align:center; margin-top:20px; padding:12px;
        background:#f0f4f0; border-radius:10px;
        font-size:12px; color:#555;">
      <strong>Input:</strong> Whisper base.en (STT) or typed text &nbsp;|&nbsp;
      <strong>Retrieval:</strong> multilingual-e5-base + FAISS + BM25 + CrossEncoder &nbsp;|&nbsp;
      <strong>LLM:</strong> Gemma 2B-IT &nbsp;|&nbsp;
      <strong>Memory:</strong> LSTM sliding window &nbsp;|&nbsp;
      <strong>Output:</strong> Text + gTTS audio
    </div>
    """)

    # ── Events ───────────────────────────────────────────────
    submit_btn.click(
        fn=gradio_submit,
        inputs=[audio_input, text_input, history_state],
        outputs=[response_output, audio_output, history_state, history_panel],
        show_progress=True
    )
    text_input.submit(
        fn=gradio_submit,
        inputs=[audio_input, text_input, history_state],
        outputs=[response_output, audio_output, history_state, history_panel]
    )
    clear_btn.click(
        fn=clear_all,
        outputs=[
            audio_input, text_input,
            response_output, audio_output,
            history_state, history_panel
        ]
    )


print("🚀 Launching...")
demo.launch(share=True, debug=False, show_error=True)

/tmp/ipykernel_55/554241718.py:45: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


🚀 Launching...
* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://fea14d8eaa4672990c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🎤 Processing audio input...
  📂 Loading audio...
  ⏱  Duration: 8.0s
  🎤 Transcribing (English)...


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✅ Transcribed: Tell me about As-Sunnah Skill Development Institute.
  📝 Transcribed: Tell me about As-Sunnah Skill Development Institute.

📝 Question  : Tell me about As-Sunnah Skill Development Institute.
📥 Input type: voice
🔗 Memory    : 0 prior turns
🔄 Retrieving and generating...
✅ Confidence: 0.8655
✅ Fallback  : False
🔊 Generating audio response...
  🔊 TTS saved: /tmp/tts_answer_4jav4g3_.mp3
🎤 Processing audio input...
  📂 Loading audio...
  ⏱  Duration: 3.2s
  🎤 Transcribing (English)...
  ✅ Transcribed: More detailed.
  📝 Transcribed: More detailed.

📝 Question  : More detailed.
📥 Input type: voice
🔗 Memory    : 1 prior turns
🔄 Retrieving and generating...
  🔗 Expanded query: Tell me about As-Sunnah Skill Development Institute. — More detailed.


...

✅ Confidence: 0.8132
✅ Fallback  : False
🔊 Generating audio response...
  🔊 TTS saved: /tmp/tts_answer_nlsr1chn.mp3

📝 Question  : What donation funds does As-Sunnah Foundation offer, and what are their purposes?
📥 Input type: text
🔗 Memory    : 6 prior turns
🔄 Retrieving and generating...


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Confidence: 0.5820
✅ Fallback  : False
🔊 Generating audio response...
  🔊 TTS saved: /tmp/tts_answer_1dl8g5ub.mp3
🎤 Processing audio input...
  📂 Loading audio...
  ⏱  Duration: 6.3s
  🎤 Transcribing (English)...
  ✅ Transcribed: What are the core principles and values of As-Sunnah Foundation?
  📝 Transcribed: What are the core principles and values of As-Sunnah Foundation?

📝 Question  : What are the core principles and values of As-Sunnah Foundation?
📥 Input type: voice
🔗 Memory    : 6 prior turns
🔄 Retrieving and generating...


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Confidence: 0.8725
✅ Fallback  : False
🔊 Generating audio response...
  🔊 TTS saved: /tmp/tts_answer_asolndta.mp3


In [ ]:
#END